# Modified SCGTNet — Training & Evaluasi

## Cell 1 — Import Library

In [ ]:
import numpy as np
import scipy.io as sio
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Import arsitektur model
from models.modified_scgtnet import build_modified_scgtnet

print('TensorFlow version :', tf.__version__)
print('GPU tersedia        :', tf.config.list_physical_devices('GPU'))
print('Library siap!')

## Cell 2 — Konfigurasi

In [ ]:
# ── Parameter Data ────────────────────────────────────────
T         = 100
C         = 16
N_CLASSES = 52

# ── Parameter Training ────────────────────────────────────
N_FOLDS       = 5
TEST_SIZE     = 0.20
BATCH_SIZE    = 256
MAX_EPOCHS    = 200
LEARNING_RATE = 0.0001
PATIENCE      = 10
RANDOM_STATE  = 42

# ── Parameter Model ───────────────────────────────────────
MSC_FILTERS  = 8
GRU_UNITS    = 32
NUM_HEADS    = 2
DROPOUT_RATE = 0.3

# ── Path ──────────────────────────────────────────────────
DATA_PATH           = f'data/windows/{T}/all_subjects_windows.mat'
PROJECT_VERSION     = f'{T}_half'
RESULT_DIR          = f'results/{PROJECT_VERSION}'
MODEL_DIR           = f'results/{PROJECT_VERSION}/models'
FIGURE_DIR          = f'results/{PROJECT_VERSION}/figures'
DATA_DIR            = f'results/{PROJECT_VERSION}/data'
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


print('Konfigurasi:')
print(f'  Data path    : {DATA_PATH}')
print(f'  Input shape  : ({T}, {C})')
print(f'  Jumlah kelas : {N_CLASSES}')
print(f'  K-Fold       : {N_FOLDS}')
print(f'  Unseen test  : {int(TEST_SIZE*100)}%')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Max epochs   : {MAX_EPOCHS}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Early stop   : {PATIENCE} epoch')

## Cell 3 — Load Data

In [ ]:
print(f'Memuat data dari {DATA_PATH}')
data    = sio.loadmat(DATA_PATH)
windows = data['windows'].astype(np.float32)
labels  = data['labels'].flatten().astype(np.int32)

print(f'Shape windows : {windows.shape}')
print(f'Shape labels  : {labels.shape}')
print(f'Label unik    : {np.unique(labels)}')
print(f'Jumlah kelas  : {len(np.unique(labels))}')
 
# Remap label ke indeks 0-based untuk one-hot encoding
# Label asli: 1-52 → indeks: 0-51
label_asli  = np.unique(labels)
label_map   = {lama: baru for baru, lama in enumerate(label_asli)}
labels_idx  = np.array([label_map[l] for l in labels], dtype=np.int32)

print(f'\nLabel setelah remap (0-based):')
print(f'  Min: {labels_idx.min()}, Max: {labels_idx.max()}')

# Distribusi kelas
print(f'\nDistribusi sampel per kelas:')
for kelas in np.unique(labels_idx):
    jumlah = np.sum(labels_idx == kelas)
    print(f'  Kelas {kelas+1:2d}: {jumlah} window')

## Cell 4 — Pisahkan Unseen Test Set (20%)

In [ ]:
# Stratified split memastikan distribusi kelas seimbang
X_trainval, X_test, y_trainval, y_test = train_test_split(
    windows,
    labels_idx,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = labels_idx
)

print('Pembagian data:')
print(f'  Total data        : {len(windows)}')
print(f'  Train+Val (80%)   : {len(X_trainval)}')
print(f'  Unseen test(20%)  : {len(X_test)}')
print(f'  Shape X_trainval  : {X_trainval.shape}')
print(f'  Shape X_test      : {X_test.shape}')

sio.savemat(
    f'{DATA_DIR}/test.mat',
    {
        'X_test': X_test,
        'y_test': y_test
    }
)

sio.savemat(
    f'{DATA_DIR}/trainval.mat',
    {
        'X_trainval': X_trainval,
        'y_trainval': y_trainval
    }
)

## Cell 5 — Z-Scores Normalization

In [ ]:
from utils.zscore_per_window import zscore_per_window

X_trainval, mu_train, sigma_train = zscore_per_window(X_trainval)
X_test, mu_test, sigma_test = zscore_per_window(X_test)

sio.savemat(
    f'{DATA_DIR}/test_z.mat',
    {
        'X_test': X_test,
        'y_test': y_test
    }
)

sio.savemat(
    f'{DATA_DIR}/trainval_z.mat',
    {
        'X_trainval': X_trainval,
        'y_trainval': y_trainval
    }
)

# np.save(f'{DATA_DIR}/X_test.npy', X_test)
# np.save(f'{DATA_DIR}/y_test.npy', y_test)
# np.save(f'{DATA_DIR}/X_trainval.npy', X_trainval)
# np.save(f'{DATA_DIR}/y_trainval.npy', y_trainval)

## Cell 6 — Fungsi Helper

In [ ]:
def hitung_metrik(y_true, y_pred):
    """
    Menghitung accuracy, precision, recall, dan F1-score.
    Menggunakan weighted average untuk multi-class.
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    return acc, prec, rec, f1


def plot_history(history, fold, save_dir):
    """
    Plot kurva akurasi dan loss selama training.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Akurasi
    ax1.plot(history.history['accuracy'],     label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validasi')
    ax1.set_title(f'Akurasi — Fold {fold}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Akurasi')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'],     label='Train')
    ax2.plot(history.history['val_loss'], label='Validasi')
    ax2.set_title(f'Loss — Fold {fold}')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'history_fold_{fold}.png'), dpi=150)
    plt.show()
    plt.close()


def plot_confusion_matrix(y_true, y_pred, title, save_path):
    """
    Plot confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(16, 14))
    sns.heatmap(
        cm,
        annot  = False,
        fmt    = 'd',
        cmap   = 'Blues',
        xticklabels = range(1, N_CLASSES + 1),
        yticklabels = range(1, N_CLASSES + 1)
    )
    plt.title(title, fontsize=14)
    plt.xlabel('Prediksi', fontsize=12)
    plt.ylabel('Label Sebenarnya', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()


print('Fungsi helper siap!')

## Cell 7 — 5-Fold Cross Validation

In [ ]:
# Inisialisasi K-Fold
skf = StratifiedKFold(
    n_splits  = N_FOLDS,
    shuffle   = True,
    random_state = RANDOM_STATE
)

# Penyimpanan hasil setiap fold
hasil_fold = []
best_val_acc      = 0.0
best_fold         = -1
best_model_path   = ''

print('=' * 55)
print('  5-FOLD CROSS VALIDATION')
print('=' * 55)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_trainval, y_trainval), start=1
):
    print(f'\n[FOLD {fold}/{N_FOLDS}]')
    print('-' * 40)

    # ── Pembagian data fold ───────────────────────────────
    X_train, X_val = X_trainval[train_idx], X_trainval[val_idx]
    y_train, y_val = y_trainval[train_idx], y_trainval[val_idx]

    print(f'  Train : {len(X_train)} sampel')
    print(f'  Val   : {len(X_val)} sampel')

    # ── One-hot encoding ──────────────────────────────────
    y_train_oh = to_categorical(y_train, num_classes=N_CLASSES)
    y_val_oh   = to_categorical(y_val,   num_classes=N_CLASSES)

    # ── Build model baru untuk setiap fold ───────────────
    tf.keras.backend.clear_session()
    model = build_modified_scgtnet(
        T            = T,
        C            = C,
        n_classes    = N_CLASSES,
        msc_filters  = MSC_FILTERS,
        gru_units    = GRU_UNITS,
        num_heads    = NUM_HEADS,
        dropout_rate = DROPOUT_RATE
    )

    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy']
    )

    # ── Callbacks ─────────────────────────────────────────
    model_path = os.path.join(MODEL_DIR, f'model_fold_{fold}.keras')

    callbacks = [
        # Simpan model terbaik per fold
        ModelCheckpoint(
            filepath         = model_path,
            monitor          = 'val_accuracy',
            save_best_only   = True,
            mode             = 'max',
            verbose          = 0
        ),
        # Hentikan training jika val_accuracy tidak membaik
        EarlyStopping(
            monitor          = 'val_accuracy',
            patience         = PATIENCE,
            restore_best_weights = True,
            verbose          = 1
        ),
        # Kurangi learning rate jika stagnan
        ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.5,
            patience = 5,
            min_lr   = 1e-6,
            verbose  = 0
        )
    ]

    # ── Training ──────────────────────────────────────────
    history = model.fit(
        X_train, y_train_oh,
        validation_data = (X_val, y_val_oh),
        epochs          = MAX_EPOCHS,
        batch_size      = BATCH_SIZE,
        callbacks       = callbacks,
        verbose         = 1
    )

    # ── Evaluasi pada validasi ────────────────────────────
    y_val_pred_prob = model.predict(X_val, verbose=0)
    y_val_pred      = np.argmax(y_val_pred_prob, axis=1)

    acc, prec, rec, f1 = hitung_metrik(y_val, y_val_pred)

    print(f'\n  Hasil Fold {fold}:')
    print(f'    Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
    print(f'    Precision : {prec:.4f} ({prec*100:.2f}%)')
    print(f'    Recall    : {rec:.4f} ({rec*100:.2f}%)')
    print(f'    F1-Score  : {f1:.4f} ({f1*100:.2f}%)')

    # Simpan hasil fold
    hasil_fold.append({
        'fold'      : fold,
        'accuracy'  : float(acc),
        'precision' : float(prec),
        'recall'    : float(rec),
        'f1_score'  : float(f1),
        'model_path': model_path,
        'epochs'    : len(history.history['accuracy'])
    })

    # Catat model terbaik
    if acc > best_val_acc:
        best_val_acc    = acc
        best_fold       = fold
        best_model_path = model_path

    # Plot history
    plot_history(history, fold, FIGURE_DIR)

print('\n' + '=' * 55)
print('  5-FOLD SELESAI')
print('=' * 55)

## Cell 7 — Ringkasan Hasil Cross Validation

In [ ]:
# Hitung rata-rata dan standar deviasi seluruh fold
acc_list  = [h['accuracy']  for h in hasil_fold]
prec_list = [h['precision'] for h in hasil_fold]
rec_list  = [h['recall']    for h in hasil_fold]
f1_list   = [h['f1_score']  for h in hasil_fold]

print('RINGKASAN HASIL 5-FOLD CROSS VALIDATION')
print('=' * 55)
print(f'{"Fold":<8} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print('-' * 55)

for h in hasil_fold:
    print(
        f'Fold {h["fold"]:<3} '
        f'{h["accuracy"]*100:>9.2f}% '
        f'{h["precision"]*100:>9.2f}% '
        f'{h["recall"]*100:>9.2f}% '
        f'{h["f1_score"]*100:>9.2f}%'
    )

print('-' * 55)
print(
    f'{"Rata-rata":<8} '
    f'{np.mean(acc_list)*100:>9.2f}% '
    f'{np.mean(prec_list)*100:>9.2f}% '
    f'{np.mean(rec_list)*100:>9.2f}% '
    f'{np.mean(f1_list)*100:>9.2f}%'
)
print(
    f'{"Std Dev":<8} '
    f'{np.std(acc_list)*100:>9.2f}% '
    f'{np.std(prec_list)*100:>9.2f}% '
    f'{np.std(rec_list)*100:>9.2f}% '
    f'{np.std(f1_list)*100:>9.2f}%'
)
print('=' * 55)
print(f'Model terbaik : Fold {best_fold} (Accuracy: {best_val_acc*100:.2f}%)')
print(f'Model path    : {best_model_path}')

# Simpan hasil ke JSON
hasil_cv = {
    'fold_results' : hasil_fold,
    'mean_accuracy'  : float(np.mean(acc_list)),
    'std_accuracy'   : float(np.std(acc_list)),
    'mean_precision' : float(np.mean(prec_list)),
    'std_precision'  : float(np.std(prec_list)),
    'mean_recall'    : float(np.mean(rec_list)),
    'std_recall'     : float(np.std(rec_list)),
    'mean_f1'        : float(np.mean(f1_list)),
    'std_f1'         : float(np.std(f1_list)),
    'best_fold'      : best_fold,
    'best_model_path': best_model_path
}

with open(f'{RESULT_DIR}/hasil_cross_validation.json', 'w') as f:
    json.dump(hasil_cv, f, indent=2)

print('\nHasil disimpan ke results/hasil_cross_validation.json')

## Cell 8 — Evaluasi pada Unseen Test Set

In [ ]:
from layers.asa import ASAModule
from layers.msc import MSCModule
from layers.bi_gru import BiGRUModule
from layers.transformer import TransformerEncoderModule

print('EVALUASI PADA UNSEEN TEST SET')
print('=' * 55)
print(f'Menggunakan model terbaik: Fold {best_fold}')
print(f'Model path: {best_model_path}')

# Load model terbaik
best_model = tf.keras.models.load_model(
    best_model_path,
    custom_objects={
        'ASAModule'                : ASAModule,
        'MSCModule'                : MSCModule,
        'BiGRUModule'              : BiGRUModule,
        'TransformerEncoderModule' : TransformerEncoderModule
    }
)

# Load unseen test set
X_test = np.load(f'{DATA_DIR}/X_test_unseen.npy')
y_test = np.load(f'{DATA_DIR}/y_test_unseen.npy')

print(f'\nUnseen test set: {len(X_test)} sampel')

# Prediksi
y_test_pred_prob = best_model.predict(X_test, verbose=0)
y_test_pred      = np.argmax(y_test_pred_prob, axis=1)

# Hitung metrik
acc, prec, rec, f1 = hitung_metrik(y_test, y_test_pred)

print('\nHasil Evaluasi Unseen Test Set:')
print(f'  Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f} ({prec*100:.2f}%)')
print(f'  Recall    : {rec:.4f} ({rec*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f} ({f1*100:.2f}%)')

# Classification report
print('\nClassification Report:')
print(classification_report(
    y_test, y_test_pred,
    target_names=[f'Gestur {i+1}' for i in range(N_CLASSES)],
    zero_division=0
))

# Plot confusion matrix
plot_confusion_matrix(
    y_test, y_test_pred,
    title     = 'Confusion Matrix — Unseen Test Set (Modified SCGTNet)',
    save_path = os.path.join(FIGURE_DIR, 'confusion_matrix_test.png')
)

# Simpan hasil evaluasi
hasil_test = {
    'window-size'   : T,
    'step-size'     : 20,
    'msc-filters'   : MSC_FILTERS,
    'gru-units'     : GRU_UNITS,
    'num-heads'     : NUM_HEADS,
    'dropout-rate'  : DROPOUT_RATE,
    'accuracy'      : float(acc),
    'precision'     : float(prec),
    'recall'        : float(rec),
    'f1_score'      : float(f1),
    'best_fold'     : best_fold
}

with open(f'{RESULT_DIR}/hasil_unseen_test.json', 'w') as f:
    json.dump(hasil_test, f, indent=2)

print('\nHasil disimpan ke results/hasil_unseen_test.json')

## Cell 9 — Ringkasan Akhir

In [ ]:
print('RINGKASAN AKHIR — Modified SCGTNet')
print('=' * 55)
print('5-Fold Cross Validation (80% data):')
print(f'  Accuracy  : {np.mean(acc_list)*100:.2f}% ± {np.std(acc_list)*100:.2f}%')
print(f'  Precision : {np.mean(prec_list)*100:.2f}% ± {np.std(prec_list)*100:.2f}%')
print(f'  Recall    : {np.mean(rec_list)*100:.2f}% ± {np.std(rec_list)*100:.2f}%')
print(f'  F1-Score  : {np.mean(f1_list)*100:.2f}% ± {np.std(f1_list)*100:.2f}%')
print()
print('Unseen Test Set (20% data):')
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print('=' * 55)